In [ ]:
from google.colab import drive
import os
import torch

# Check GPU & PyTorch (Sanity Check)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
!nvidia-smi

# Mount Drive
drive.mount('/content/drive')

# Configuration
REPO_URL = "https://github.com/fabianandresgrob/gaussian-splatting.git"
REPO_BRANCH = "view-selection"
WORKING_DIR = "/content/gsplat_workspace"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

# 4. Create Workspace & Clone
os.makedirs(WORKING_DIR, exist_ok=True)
os.chdir(WORKING_DIR)

if not os.path.exists(REPO_NAME):
    print(f"--- Cloning {REPO_NAME} ---")
    !git clone {REPO_URL} --recursive
    os.chdir(REPO_NAME)
    !git checkout {REPO_BRANCH}
else:
    print(f"--- Repository exists. Pulling latest (forced) ---")
    os.chdir(REPO_NAME)
    # We don't care about local changes, just overwrite
    !git reset --hard
    !git pull --force
    !git checkout {REPO_BRANCH}
    !git submodule update --init --recursive

REPO_PATH = os.getcwd()
print(f"SUCCESS: Repo ready at {REPO_PATH}")

In [ ]:
import setup_utils
import experiment_lib
import pandas as pd
import numpy as np
import os

# 1. Install Dependencies and compile extensions
setup_utils.setup_environment(os.getcwd())

In [ ]:
# Install tensorboard first
!pip install tensorboard -q

OUTPUT_ROOT = "/content/drive/MyDrive/3DGS_Results" 

# Launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_ROOT}

In [ ]:
# 2. Define Experiments
DATA_ROOT = "/content/drive/MyDrive/Scannet++/data/scenes/data"

SCENES = ["0c5385e84b"]  # Add more scenes
STRATEGIES = ["random", "fixed_prob"]  # Add different strats like "epoch_based", "clustering", "no_replace"
SEEDS = [0, 1, 2, 3, 4]

# Training configuration
TRAINING_CONFIG = {
    "iterations": 10000,  # Total training iterations
    "test_iterations": [1000, 2000, 4000, 6000, 8000, 10000], # Test at multiple points
    "save_iterations": [10000], # Save at the end of training
    "data_device": "cuda",
    "resolution": 2,  # Half resolution for faster training
}

# Strategy-specific configurations, not all strategies need configs
STRATEGY_CONFIGS = {
    "random": "{}",
    "fixed_prob": '{"temperature": 1.0, "distance_weight": 0.5, "diversity_weight": 0.5}',
    "epoch_based": '{"penalty_strength": 1.0}',
    "clustering": '{"n_clusters": 10, "temperature": 1.0}',
    "no_replace": "{}",
}

# 3. Run Experiments
current_repo_path = os.getcwd()

for scene in SCENES:
    scene_path = os.path.join(DATA_ROOT, scene, "dslr")
    
    for strategy in STRATEGIES:
        exp_name = f"{scene}_{strategy}"
        view_config = STRATEGY_CONFIGS.get(strategy, "{}")
        
        for seed in SEEDS:
            experiment_lib.run_training(
                repo_path=current_repo_path,
                data_path=scene_path,
                output_dir=OUTPUT_ROOT,
                strategy=strategy,
                seed=seed,
                exp_name=exp_name,
                iterations=TRAINING_CONFIG["iterations"],
                test_iterations=TRAINING_CONFIG["test_iterations"],
                save_iterations=TRAINING_CONFIG["save_iterations"],
                data_device=TRAINING_CONFIG["data_device"],
                view_selection_config=view_config,
                resolution=TRAINING_CONFIG["resolution"],
            )

In [ ]:
# 4. Aggregate Results Across All Experiments
results = []
for scene in SCENES:
    for strategy in STRATEGIES:
        rep = experiment_lib.aggregate_results(OUTPUT_ROOT, scene, strategy)
        if rep: 
            results.append(rep)

df = pd.DataFrame(results)
print("\n" + "="*70)
print("FINAL AGGREGATED REPORT (Mean +/- Std across seeds)")
print("="*70)
print(df.to_string(index=False))
df.to_csv(os.path.join(OUTPUT_ROOT, "final_experiment_report.csv"), index=False)
print(f"\n[Saved] {os.path.join(OUTPUT_ROOT, 'final_experiment_report.csv')}")

In [ ]:
# 5. Detailed Per-Seed Analysis
import json
import glob

def load_all_results(output_root, scenes, strategies):
    """Load all individual seed results into a detailed DataFrame."""
    all_data = []
    
    for scene in scenes:
        for strategy in strategies:
            exp_folder = os.path.join(output_root, f"{scene}_{strategy}")
            seed_dirs = sorted(glob.glob(os.path.join(exp_folder, "seed_*")))
            
            for seed_dir in seed_dirs:
                seed = int(os.path.basename(seed_dir).split("_")[1])
                
                # Load final results
                final_file = os.path.join(seed_dir, "final_results.json")
                if os.path.exists(final_file):
                    with open(final_file, 'r') as f:
                        final = json.load(f)
                    
                    # Load metrics history for peak test PSNR
                    history_file = os.path.join(seed_dir, "metrics_history.json")
                    peak_psnr = final['mean_psnr']
                    peak_iter = None
                    
                    if os.path.exists(history_file):
                        with open(history_file, 'r') as f:
                            history = json.load(f)
                        if history:
                            test_psnrs = [(h['iteration'], h['test']['PSNR']) for h in history if 'test' in h]
                            if test_psnrs:
                                peak_iter, peak_psnr = max(test_psnrs, key=lambda x: x[1])
                    
                    all_data.append({
                        'Scene': scene,
                        'Strategy': strategy,
                        'Seed': seed,
                        'Final_PSNR': final['mean_psnr'],
                        'Final_SSIM': final['mean_ssim'],
                        'Final_LPIPS': final['mean_lpips'],
                        'Peak_PSNR': peak_psnr,
                        'Peak_Iter': peak_iter,
                    })
    
    return pd.DataFrame(all_data)

# Load all results
df_detailed = load_all_results(OUTPUT_ROOT, SCENES, STRATEGIES)
print("Per-Seed Results:")
print(df_detailed.to_string(index=False))
print(f"\nTotal runs: {len(df_detailed)}")

In [ ]:
# 6. Statistical Comparison Between Strategies
def compare_strategies(df_detailed):
    """Compare strategies with proper statistics."""
    print("\n" + "="*70)
    print("STRATEGY COMPARISON (per scene)")
    print("="*70)
    
    for scene in df_detailed['Scene'].unique():
        print(f"\n[Scene] {scene}")
        print("-" * 50)
        
        scene_data = df_detailed[df_detailed['Scene'] == scene]
        
        # Group by strategy
        summary = scene_data.groupby('Strategy').agg({
            'Final_PSNR': ['mean', 'std'],
            'Peak_PSNR': ['mean', 'std'],
            'Final_SSIM': ['mean', 'std'],
            'Final_LPIPS': ['mean', 'std'],
            'Peak_Iter': 'mean',
        }).round(4)
        
        # Flatten column names
        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        
        # Format nicely
        for strategy in summary.index:
            row = summary.loc[strategy]
            print(f"\n  Strategy: {strategy}")
            print(f"    Final PSNR:  {row['Final_PSNR_mean']:.3f} +/- {row['Final_PSNR_std']:.3f}")
            print(f"    Peak PSNR:   {row['Peak_PSNR_mean']:.3f} +/- {row['Peak_PSNR_std']:.3f} (at iter {row['Peak_Iter_mean']:.0f})")
            print(f"    Final SSIM:  {row['Final_SSIM_mean']:.4f} +/- {row['Final_SSIM_std']:.4f}")
            print(f"    Final LPIPS: {row['Final_LPIPS_mean']:.4f} +/- {row['Final_LPIPS_std']:.4f}")
        
        # Find best strategy
        best_strategy = summary['Peak_PSNR_mean'].idxmax()
        print(f"\n  >> Best strategy (by Peak PSNR): {best_strategy}")

compare_strategies(df_detailed)

In [ ]:
# 7. Plot Training Curves (PSNR over iterations)
import matplotlib.pyplot as plt

def plot_training_curves(output_root, scenes, strategies, metric='PSNR'):
    """Plot training curves for all strategies."""
    fig, axes = plt.subplots(1, len(scenes), figsize=(7*len(scenes), 5))
    if len(scenes) == 1:
        axes = [axes]
    
    colors = {'random': 'blue', 'fixed_prob': 'orange', 'epoch_based': 'green', 
              'clustering': 'red', 'no_replace': 'purple'}
    
    for ax, scene in zip(axes, scenes):
        ax.set_title(f'Scene: {scene}')
        ax.set_xlabel('Iteration')
        ax.set_ylabel(f'Test {metric}')
        
        for strategy in strategies:
            exp_folder = os.path.join(output_root, f"{scene}_{strategy}")
            seed_dirs = sorted(glob.glob(os.path.join(exp_folder, "seed_*")))
            
            all_curves = []
            for seed_dir in seed_dirs:
                history_file = os.path.join(seed_dir, "metrics_history.json")
                if os.path.exists(history_file):
                    with open(history_file, 'r') as f:
                        history = json.load(f)
                    if history:
                        iters = [h['iteration'] for h in history if 'test' in h]
                        values = [h['test'][metric] for h in history if 'test' in h]
                        all_curves.append((iters, values))
            
            if all_curves:
                # Plot individual seeds with transparency
                for iters, values in all_curves:
                    ax.plot(iters, values, color=colors.get(strategy, 'gray'), 
                           alpha=0.2, linewidth=1)
                
                # Plot mean curve
                # Align curves by iteration
                all_iters = all_curves[0][0]
                mean_values = np.mean([[v for v in curve[1]] for curve in all_curves], axis=0)
                ax.plot(all_iters, mean_values, color=colors.get(strategy, 'gray'), 
                       linewidth=2, label=strategy)
        
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_root, 'training_curves.png'), dpi=150)
    plt.show()
    print(f"[Saved] {os.path.join(output_root, 'training_curves.png')}")

plot_training_curves(OUTPUT_ROOT, SCENES, STRATEGIES)

In [ ]:
# 8b. Plot Loss Curves from TensorBoard Logs
from tensorboard.backend.event_processing import event_accumulator

def read_tensorboard_scalars(log_dir, tag):
    """Read scalar values from TensorBoard event files."""
    ea = event_accumulator.EventAccumulator(log_dir)
    ea.Reload()
    
    if tag not in ea.Tags()['scalars']:
        return [], []
    
    events = ea.Scalars(tag)
    steps = [e.step for e in events]
    values = [e.value for e in events]
    return steps, values

def plot_loss_curves(output_root, scenes, strategies):
    """Plot training loss curves from TensorBoard logs."""
    fig, axes = plt.subplots(1, len(scenes), figsize=(7*len(scenes), 5))
    if len(scenes) == 1:
        axes = [axes]
    
    colors = {'random': 'blue', 'fixed_prob': 'orange', 'epoch_based': 'green', 
              'clustering': 'red', 'no_replace': 'purple'}
    
    for ax, scene in zip(axes, scenes):
        ax.set_title(f'Training Loss: {scene}')
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Total Loss')
        ax.set_yscale('log')  # Log scale for loss
        
        for strategy in strategies:
            exp_folder = os.path.join(output_root, f"{scene}_{strategy}")
            seed_dirs = sorted(glob.glob(os.path.join(exp_folder, "seed_*")))
            
            all_curves = []
            for seed_dir in seed_dirs:
                try:
                    steps, values = read_tensorboard_scalars(
                        seed_dir, 'train_loss_patches/total_loss'
                    )
                    if steps:
                        all_curves.append((steps, values))
                except Exception as e:
                    print(f"Warning: Could not read TB logs from {seed_dir}: {e}")
            
            if all_curves:
                # Plot individual seeds with transparency
                for steps, values in all_curves:
                    ax.plot(steps, values, color=colors.get(strategy, 'gray'), 
                           alpha=0.15, linewidth=0.5)
                
                # Compute and plot mean (subsample for speed)
                min_len = min(len(c[0]) for c in all_curves)
                mean_values = np.mean([c[1][:min_len] for c in all_curves], axis=0)
                ax.plot(all_curves[0][0][:min_len], mean_values, 
                       color=colors.get(strategy, 'gray'), linewidth=2, label=strategy)
        
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_root, 'loss_curves.png'), dpi=150)
    plt.show()
    print(f"[Saved] {os.path.join(output_root, 'loss_curves.png')}")

plot_loss_curves(OUTPUT_ROOT, SCENES, STRATEGIES)

In [ ]:
# 8. Bar Chart Comparison
def plot_bar_comparison(df_detailed, metric='Peak_PSNR'):
    """Create bar chart comparing strategies across scenes."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Compute mean and std per scene+strategy
    summary = df_detailed.groupby(['Scene', 'Strategy'])[metric].agg(['mean', 'std']).reset_index()
    
    scenes = summary['Scene'].unique()
    strategies = summary['Strategy'].unique()
    x = np.arange(len(scenes))
    width = 0.8 / len(strategies)
    
    colors = {'random': '#1f77b4', 'fixed_prob': '#ff7f0e', 'epoch_based': '#2ca02c', 
              'clustering': '#d62728', 'no_replace': '#9467bd'}
    
    for i, strategy in enumerate(strategies):
        data = summary[summary['Strategy'] == strategy]
        means = [data[data['Scene'] == s]['mean'].values[0] if s in data['Scene'].values else 0 for s in scenes]
        stds = [data[data['Scene'] == s]['std'].values[0] if s in data['Scene'].values else 0 for s in scenes]
        
        bars = ax.bar(x + i*width, means, width, label=strategy, 
                     color=colors.get(strategy, 'gray'), yerr=stds, capsize=3)
    
    ax.set_xlabel('Scene')
    ax.set_ylabel(metric.replace('_', ' '))
    ax.set_title(f'{metric.replace("_", " ")} by Strategy (Mean ± Std)')
    ax.set_xticks(x + width * (len(strategies)-1) / 2)
    ax.set_xticklabels(scenes, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_ROOT, f'{metric}_comparison.png'), dpi=150)
    plt.show()

plot_bar_comparison(df_detailed, 'Peak_PSNR')
plot_bar_comparison(df_detailed, 'Final_LPIPS')

In [ ]:
# 9. Export LaTeX Table for Paper
def export_latex_table(df_detailed, output_path):
    """Export results as LaTeX table for paper."""
    
    # Aggregate by Scene and Strategy
    summary = df_detailed.groupby(['Scene', 'Strategy']).agg({
        'Peak_PSNR': ['mean', 'std'],
        'Final_SSIM': ['mean', 'std'],
        'Final_LPIPS': ['mean', 'std'],
    }).round(3)
    
    # Create LaTeX formatted strings
    latex_rows = []
    latex_rows.append(r"\begin{table}[h]")
    latex_rows.append(r"\centering")
    latex_rows.append(r"\caption{View Selection Strategy Comparison}")
    latex_rows.append(r"\begin{tabular}{llccc}")
    latex_rows.append(r"\toprule")
    latex_rows.append(r"Scene & Strategy & PSNR $\uparrow$ & SSIM $\uparrow$ & LPIPS $\downarrow$ \\")
    latex_rows.append(r"\midrule")
    
    for (scene, strategy), row in summary.iterrows():
        psnr_str = f"{row[('Peak_PSNR', 'mean')]:.2f} +/- {row[('Peak_PSNR', 'std')]:.2f}"
        ssim_str = f"{row[('Final_SSIM', 'mean')]:.3f} +/- {row[('Final_SSIM', 'std')]:.3f}"
        lpips_str = f"{row[('Final_LPIPS', 'mean')]:.3f} +/- {row[('Final_LPIPS', 'std')]:.3f}"
        latex_rows.append(f"{scene} & {strategy} & {psnr_str} & {ssim_str} & {lpips_str} \\\\")
    
    latex_rows.append(r"\bottomrule")
    latex_rows.append(r"\end{tabular}")
    latex_rows.append(r"\label{tab:view_selection}")
    latex_rows.append(r"\end{table}")
    
    latex_str = "\n".join(latex_rows)
    
    # Save to file
    with open(output_path, 'w') as f:
        f.write(latex_str)
    
    print("LaTeX Table:")
    print(latex_str)
    print(f"\n[Saved] {output_path}")

export_latex_table(df_detailed, os.path.join(OUTPUT_ROOT, 'results_table.tex'))

Results Directory Structure
===========================
```
/content/drive/MyDrive/3DGS_Results/       <-- OUTPUT_ROOT
│
├── 0c5385e84b_random/                     <-- Experiment Folder ({scene}_{strategy})
│   ├── seed_0/                            <-- Run Folder (Per Seed)
│   │   ├── point_cloud/
│   │   │   └── iteration_30000/
│   │   │       └── point_cloud.ply        <-- Final trained Gaussian model
│   │   ├── eval/
│   │   │   ├── 00000.png                  <-- Rendered test image 0
│   │   │   ├── 00001.png
│   │   │   └── ...
│   │   ├── combine/
│   │   │   ├── 00000.png                  <-- Side-by-side: Render vs. GT
│   │   │   └── ...
│   │   ├── metrics_history.json           <-- Training curve data (Loss/PSNR over time)
│   │   ├── final_results.json             <-- Final averaged metrics for this seed
│   │   ├── console_log.txt                <-- Full training log output
│   │   ├── cameras.json                   <-- Camera parameters used
│   │   ├── cfg_args                       <-- Arguments used for this run
│   │   ├── chkpnt30000.pth                <-- PyTorch checkpoint
│   │   └── events.out.tfevents...         <-- Tensorboard logs
│   │
│   ├── seed_1/
│   │   └── ... (Same structure)
│   └── ...
│
├── 0c5385e84b_fixed_prob/                 <-- Next Experiment
│   └── ...
│
└── experiment_summary.csv                 <-- Final aggregated report of all runs
```